# AI Benchmarks & LLM Model Selection - Study Notes

> A consolidated note covering LLM model selection strategy and the modern AI
> benchmark landscape (GPQA, MMLU-Pro, HLE, LiveBench, SEAL, Vellum, LM Arena).
> Focus: RAG, QLoRA fine-tuning, and Agent engineering.

## Table of Contents

1. [Choosing the Right LLM & Model Selection Strategy](#1-choosing-the-right-llm--model-selection-strategy)
2. [Understanding AI Model Benchmarks: GPQA, MMLU-Pro, and HLE](#2-understanding-ai-model-benchmarks-gpqa-mmlu-pro-and-hle)
3. [Limitations of AI Benchmarks: Data Contamination and Overfitting](#3-limitations-of-ai-benchmarks-data-contamination-and-overfitting)
4. [Vellum, SEAL, and LiveBench - and How They Differ](#4-vellum-seal-and-livebench--and-how-they-differ)
5. [LM Arena: Blind Testing with Community Elo Ratings](#5-lm-arena-blind-testing-with-community-elo-ratings)

# 1. Choosing the Right LLM & Model Selection Strategy

## 1.1 Define Your Requirements First

Before evaluating any model, nail down these axes:

| Dimension | Questions to Ask |
|---|---|
| **Task type** | Generation, classification, extraction, reasoning, code, multimodal? |
| **Latency** | Real-time (<500ms)? Batch okay? |
| **Throughput** | Requests/sec, concurrent users? |
| **Cost** | Per-token budget, infra budget? |
| **Privacy** | Can data leave your infra? On-prem required? |
| **Context window** | How long are your inputs/outputs? |

## 1.2 The Decision Tree


    Is data privacy/on-prem required?
    ├── YES -> Open-source: LLaMA 3, Mistral, Phi-3, Qwen 2.5
    └── NO  -> Consider APIs: Claude, GPT-4o, Gemini

    What is your primary task?
    ├── Reasoning/math     -> DeepSeek-R1, Claude 3.5 Sonnet, Gemini 2.0
    ├── Code               -> Qwen2.5-Coder, DeepSeek-Coder, GPT-4o
    ├── Long context RAG   -> Claude (200k), Gemini (1M), LLaMA 3.1 (128k)
    ├── Low-cost/fast      -> Haiku 4.5, GPT-4o-mini, Phi-3-mini
    └── Fine-tuning needed -> Mistral 7B, LLaMA 3.1 8B, Qwen 2.5 7B

## 1.3 Model Size vs. Use Case

| Size | Examples | Best For |
|---|---|---|
| 1-4B | Phi-3-mini, Gemma 2B | Edge, embedded, fast inference |
| 7-13B | Mistral 7B, LLaMA 3.1 8B | Fine-tuning, on-prem, RAG |
| 30-70B | LLaMA 3.1 70B, Qwen 72B | High quality, local if possible |
| 405B+ | LLaMA 3.1 405B, GPT-4o | Max capability, API only |

## 1.4 Key Evaluation Benchmarks

* **MMLU** - general knowledge, reasoning
* **HumanEval / MBPP** - coding ability
* **MT-Bench** - instruction following, multi-turn
* **HELM** - holistic across many tasks
* **RAGBench / BEIR** - retrieval-specific tasks
* **TruthfulQA** - hallucination resistance

> Don't rely solely on benchmarks - **evaluate on YOUR data.**

## 1.5 Strategy for RAG vs. Fine-Tuning vs. Agents

Knowledge is external/changing?

    ├── YES -> Use RAG - smaller base model is fine (7B-13B)
    └── NO
        Task needs domain adaptation (style, format, specialized knowledge)?
        └── YES -> Use QLoRA fine-tuning - Mistral 7B / LLaMA 3.1 8B
        
        Task requires tool use, planning, multi-step reasoning?
        └── YES -> Use Agents - needs strong instruction-following (Claude, GPT-4o, LLaMA 70B+)

## 1.6 Practical Selection Checklist

- [ ] Run benchmark on task-specific eval set
- [ ] Measure TTFT (time to first token) and throughput
- [ ] Estimate monthly cost at expected volume
- [ ] Check license (commercial use: Apache 2.0, MIT > LLaMA custom)
- [ ] Test context length with real document sizes
- [ ] Check if model supports function/tool calling
- [ ] Verify quantization quality (GGUF Q4_K_M vs Q8 tradeoff)

## 1.7 Quick Reference for Common Scenarios

| Scenario | Recommended Model |
|---|---|
| RAG chatbot, cloud | Claude 3.5 Haiku or GPT-4o-mini |
| RAG, on-prem, fine-tuned | LLaMA 3.1 8B + QLoRA |
| Agentic pipeline | Claude Sonnet 5 or GPT-4o |
| Code generation | Qwen2.5-Coder 7B or DeepSeek-Coder |
| Document QA (long context) | Gemini 1.5 Pro or Claude |
| Low-latency production | Phi-3-mini or Haiku 4.5 |

> **Bottom line:** Start with the smallest model that passes your eval threshold - 
> scale up only when quality demands it. Cost and latency compound fast in production.

# 2. Understanding AI Model Benchmarks: GPQA, MMLU-Pro, and HLE

## 2.1 Why Benchmarks Exist (and Why They Keep Changing)

Benchmarks are **standardized tests for LLMs** - a way to measure and compare model capability objectively. But there's a problem that drives the whole story:

The Benchmark Lifecycle:

1. New benchmark released -> models score low (e.g., 30%)
2. Models improve rapidly  -> scores climb
3. Scores hit ~90%+         -> benchmark is "saturated"
4. Benchmark becomes useless -> need a HARDER one
5. Repeat.

MMLU (2020)     ->  saturated ~2023 (models hit 88-90%)

│

MMLU-Pro (2024) ->  harder replacement

│

GPQA (2023)     ->  "Google-proof" expert questions

│

HLE (2025)      ->  "Humanity's Last Exam" - the frontier

## 2.2 MMLU-Pro

**Paper:** MMLU-Pro (TIGER-Lab, 2024). The original MMLU (Massive Multitask Language Understanding, 2020) had 57 subjects and ~14,000 multiple-choice questions with 4 options each. By 2023-2024 top models scored 86-90% and it became saturated.

| Change | MMLU | MMLU-Pro |


| Answer options | 4 choices | 10 choices |
| Difficulty | Mixed | Hard only |
| Reasoning needed | Often recall | Multi-step |
| Bad questions | Some | Cleaned out |
| Question count | ~14,000 | ~12,000 |

**Why 10 options matters:** random guessing baseline drops from 25% -> 10%, so luck contributes far less and real ability is measured.

**Score impact:** the SAME model can drop ~23 points (e.g., 86% -> 63%), re-opening headroom to distinguish top models.

## 2.3 GPQA

GPQA = Graduate-Level Google-Proof Q&A (2023)
* "Graduate-Level" -> PhD-level difficulty
* "Google-Proof"  -> you can't just search for the answer
* "Q&A"          -> question-answering format

**Paper:** Rein et al., 2023. 448 questions across Biology, Physics, Chemistry, written by PhD-level experts, designed to be un-searchable.

**"Google-Proof" validation:**

| Who answers | Accuracy |
|---|---|
| PhD expert (in-domain) | 65% |
| PhD expert (out-of-domain) | ~34% |
| Non-expert + Google | 34% |
| Random guess (4 options) | 25% |

**GPQA Diamond** = the hardest, highest-quality subset (198 questions) - the version most model reports cite.


## 2.4 HLE (Humanity's Last Exam)

**Created by:** Center for AI Safety (CAIS) + Scale AI, 2025.

- ~2,500 curated questions across 100+ academic subjects
- Contributed by ~1,000 expert professors/researchers from 500+ institutions
- Both multiple-choice AND exact-answer formats; includes multi-modal questions
- When released, top frontier models scored just **3%-10%** (intentional - leaves years of measurement headroom)

**Anti-gaming design:** exact verifiable answers, a held-out private set (prevents contamination), and multiple rounds of expert review.

## 2.5 Side-by-Side Comparison

| | MMLU-Pro | GPQA | HLE |
|---|---|---|---|
| Released | 2024 | 2023 | 2025 |
| Questions | ~12,000 | 448 | ~2,500 |
| Difficulty | Hard | PhD-level | Frontier |
| Domains | 57 broad | 3 sciences | 100+ subjects |
| Format | 10-choice | 4-choice | Mixed + multimodal |
| Top scores | ~75-80% | ~60-70% | ~10-25% |

## 2.6 How to Read These Wisely

1. Higher isn't always what YOU need - benchmarks ≠ your use case.
2. Watch for **contamination** - prefer private held-out sets.
3. Compare on the **same** benchmark version (MMLU ≠ MMLU-Pro).
4. Small benchmarks = high variance (GPQA has only 448 questions).
5. Match benchmark to capability (reasoning -> GPQA/MMLU-Pro/HLE; coding -> HumanEval/SWE-bench; RAG -> BEIR/RAGBench).

**Relevance to your work:** For RAG, prioritize instruction-following, long-context handling, and faithfulness over raw benchmark scores. For QLoRA,


use benchmarks to pick a strong base then RE-benchmark on your own eval set.
For Agents, also check SWE-bench, WebArena/GAIA, t-bench.

# 3. Limitations of AI Benchmarks: Data Contamination and Overfitting

**Does it matter? Yes - critically.** These two problems are the biggest reasons never to trust a leaderboard number blindly.

What a benchmark CLAIMS to measure:
    "How well can this model REASON about NEW problems?"
What it sometimes ACTUALLY measures:
    "How well did this model MEMORIZE the test answers?"

1. DATA CONTAMINATION – the model already SAW the test
2. OVERFITTING        – the model was TUNED to beat the test

## 3.1 Data Contamination (test-set leakage)

The benchmark's questions/answers accidentally ended up inside the model's training data - the model is **remembering**, not reasoning.

3. **Canary strings** - unique embedded strings; reproducing them = it saw the data.
4. **Perturbation test** - reword/reorder; if score collapses, it memorized surface form.

## 3.2 Overfitting to Benchmarks ("teaching to the test")

Developers repeatedly tune a model to maximize a specific benchmark until it's great at that test but not the real capability. This is **Goodhart's Law:**

> "When a measure becomes a target, it ceases to be a good measure."

Overfitting creeps in subtly: picking the highest-scoring checkpoint, tweaking prompts/data mix while watching the score, adding benchmark-like data, and the whole industry optimizing the same few leaderboards.

## 3.3 Contamination vs. Overfitting

| | Contamination | Overfitting |
|---|---|---|
| Cause | Exact test answers leaked into training | Model iteratively tuned to win the test |
| Nature | Accidental memorization | Deliberate/drift-based optimization |
| Symptom | Both - high benchmark score, weak real-world performance |

## 3.4 Documented Consequences

- Inflated leaderboards – you deploy a model that underperforms.
- False model comparisons – you pick the contaminated model over a better one.
- Wasted fine-tuning effort – your QLoRA base was weaker than its scores implied.
- **GSM8k - GSM1k example:** rebuilding a math benchmark with fresh questions of equal difficulty caused some models to **drop 8-13 points** - exposing contamination/overfitting.

## 3.5 How the Field Fights Back

| Defense | How it helps |
|---|---|
| Private/held-out sets (HLE, SEAL) | Questions never published - can't leak |
| Dynamic benchmarks (LiveBench) | New questions each round - nothing to memorize |



| Canary strings | Detect if dataset was seen |
| Perturbation testing (GSM1k-style) | Exposes memorization |
| Contamination audits | N-gram/overlap checks before trusting scores |
| Human preference arenas (LMArena) | Live votes on fresh prompts – hard to game |

## 3.6 The One Rule That Solves Most of This

> **BUILD YOUR OWN PRIVATE EVALUATION SET** – from your real data, never
> published, never in training. Public benchmarks = rough filter; your private
> eval = the real decision. For QLoRA especially: split BEFORE training, keep
> eval locked away, and use early stopping when eval stalls while train loss drops.

---

# 4. Vellum, SEAL, and LiveBench – and How They Differ

## 4.1 The Key Distinction

GROUP A: STATIC DATASETS (the "test questions")
- MMLU-Pro, GPQA, HLE = fixed collections of Q&A = "the exam paper itself"

GROUP B: PLATFORMS / LIVE SYSTEMS (the "testing lab")
- Vellum, SEAL, LiveBench = systems that RUN and REPORT evaluations

## 4.2 LiveBench

**LiveBench (2024, Abacus.AI + researchers incl. Yann LeCun)** – designed to be **contamination-free** by using questions that didn't exist when models were trained.

- New questions released monthly from recent sources (news, arXiv, math competitions, fresh coding problems) – model cannot have seen them.
- Categories: math, coding, reasoning, language, data analysis, instruction-following.
- Objective, verifiable, auto-scored answers (no gameable LLM judge).
- Contamination is prevented **by design**, not just hoped for.
## 4.3 SEAL

**SEAL = Safety, Evaluations and Alignment Lab (Scale AI, 2024)** – expert-curated, **private** benchmarks.

- Questions created by experts and **never published** – only rankings are released.
- Impossible to train on or overfit to questions you can't see.
- Domains: coding, instruction-following, math, multilinguality, adversarial robustness, tool use. Periodically refreshed; neutral third-party rankings.
- Solves **both** contamination (private) and overfitting (hidden test).

## 4.4 Vellum – The Different One

**Vellum is NOT primarily a benchmark. It's a developer platform** for building, testing, and evaluating LLM applications.

- Prompt engineering (test/version prompts across models)
- Evaluation suites (build your OWN test cases with YOUR data)
- Workflow building (chain prompts, tools, RAG steps)
- Production monitoring (quality, latency, cost)

Vellum also publishes a public **LLM Leaderboard** that *aggregates* scores from other benchmarks – a "benchmark dashboard," not a benchmark itself.

## 4.5 Full Comparison

| Type | Key Trait | Contamination Resistance |
|---|---|---|
| MMLU-Pro | Static dataset | Broad 10-option QA | LOW (public) |
| GPQA | Static dataset | PhD science, Google-proof | LOW (public) |
| HLE | Static + private | Frontier expert exam | MEDIUM-HIGH (private set) |
| LiveBench | Dynamic benchmark | Fresh Qs monthly | HIGH (post-cutoff Qs) |
| SEAL | Private leaderboard | Expert, held-out | HIGH (secret Qs) |
| Vellum | Platform + aggregator | Build YOUR own evals | N/A (it's a tool) |


## 4.6 The Evolution Story

STAGE 1: Static public benchmarks (MMLU-Pro, GPQA, HLE)

↳ contamination + overfitting

STAGE 2: Contamination-resistantLiveBench - refresh questions │ SEAL - hide questions

STAGE 3: Measure YOUR taskVellum - evaluate on your own data

This maps exactly onto the contamination defenses: "use dynamic benchmarks" -> LiveBench; "use private held-out sets" -> SEAL/HLE; "build your own eval" -> Vellum.

---

# 5. LM Arena: Blind Testing with Community Elo Ratings

## 5.1 What It Is

**LM Arena (formerly "Chatbot Arena", LMSYS + UC Berkeley)** – a public platform where **real humans** compare two anonymous models side-by-side and vote for the better response. Votes aggregate into an **Elo** ranking (from chess).

> It answers a different question: not "which model is most correct on a test?"
> but **"which model do real people actually prefer using?"**

## 5.2 Blind Testing

BLIND = the voter does NOT know which models they're judging.

1. You type any prompt.
2. Two anonymous models answer: "Model A" and "Model B".
3. You vote: [A better] [B better] [Tie] [Both bad].
4. ONLY AFTER VOTING – the model names are revealed.

**Why blind matters:** removes **brand bias** – you judge response quality, not reputation. Same principle as blind wine tastings and drug trials.

**Pairwise comparison:** compare two at a time (A vs B). Humans are bad at absolute scores but reliable at relative choices.

## 5.3 Elo Rating

Each model is treated like a chess player; each vote is a "match."

**Expected win probability of A over B:**

$$E_A = \frac{1}{1 + 10^{(R_B - R_A)/400}}$$

| Rating Gap | Stronger model's expected win rate |
|---|---|
| 0 points | 50% (coin flip) |
| 100 points | 64% |
| 200 points | 76% |
| 400 points | 91% |
| 800 points | 99% |

**Update rule after each vote:**

$$R_{\text{new}} = R_{\text{old}} + K \times (S - E)$$

* $S$ = actual result (1 win, 0 loss, 0.5 tie)
* $E$ = expected result (from formula)
* $K$ = K-factor (step size)

![Elo Expected Win Probability Curve](image-10.png)

**Worked example** (both rated 1200, K=32, A wins):

$E_A = 0.5, S_A = 1$

$R_{A,\text{new}} = 1200 + 32 \times (1 - 0.5) = 1216$

$R_{B,\text{new}} = 1200 + 32 \times (0 - 0.5) = 1184$

Key insight: a low-rated model beating a high-rated one gains many points (big upset = big update); the favorite winning gains few.

## 5.4 Modern Refinement - Bradley-Terry

LM Arena moved from sequential Elo to the **Bradley-Terry** model: computes ratings from ALL votes at once (order-independent, more stable) – still loosely called "Elo." Leaderboard shows **confidence intervals** (e.g. $1287 \pm 5$); models with overlapping intervals are **statistically tied**.

## 5.5 Terminology Glossary

| Term | Meaning |
|---|---|
| Blind test | Voter doesn't know which models -> removes bias |
| Pairwise comparison | Compare two models head-to-head, not scored alone |
| Battle / Match | A single A-vs-B comparison + vote |
| Elo rating | Number ranking each model by win/loss history |
| Expected score (E) | Predicted win probability from the rating gap |
| K-factor | How much a single vote moves the rating |
| Bradley-Terry | Order-independent method now used (still called "Elo") |
| Confidence interval | $\pm$ uncertainty; overlapping = statistically tied |
| Win rate | % of battles a model won overall |

## 5.6 Strengths and Limitations

**Strengths:** contamination-proof (real-time user prompts, no fixed test set), measures real preference (helpfulness, tone, clarity, formatting), hard to game, huge sample size.

**Limitations:**
- **Popularity ≠ correctness** – humans may prefer a confident but subtly wrong answer.
- **Style bias** – longer/prettier answers often win (Arena added a style-controlled ranking).
- **Prompt distribution skew** – reflects casual chat, not your specialized domain.
- **Not task-specific** – high Arena rank ≠ good on your RAG/fine-tuned domain.

## 5.7 How It Fits With Other Benchmarks

- **MMLU-Pro / GPQA / HLE** – "Is it CORRECT?" (static tests)
- **LiveBench** – "Correct, contamination-free?"
- **SEAL** – "Correct, judged privately?"
- **LM Arena** – "Do HUMANS PREFER it?" – unique
- **Vellum** – "Does it work for MY task?" (tool)

A great model should score well on **both** objective benchmarks and human preference. High Arena + low benchmarks ➔ "charming but not always right"; high benchmarks + low Arena ➔ "smart but unpleasant to use."

---

## # Overall Takeaways

- Match the **model to the task, latency, cost, and privacy** constraints – smallest model that passes your eval wins.
- Benchmarks evolve because they **saturate**: MMLU ➔ MMLU-Pro ➔ GPQA ➔ HLE.
- **Contamination and overfitting** are the central reasons leaderboard numbers mislead.
- Modern systems fight back: **LiveBench** (fresh questions), **SEAL** (private questions), **LM Arena** (blind human preference).
- **Vellum** is a platform to build your own evals – the productized version of the golden rule.
- **Golden rule:** public benchmarks are a rough filter; **your own private eval set on your own data is the real decision** – critical for RAG faithfulness and QLoRA fine-tuning.

---

\*Note: benchmark scores are approximate snapshots and shift as models improve;

